## Clean Names, Leave Page Name Only

In [1]:
import os

Get all files in scraped HTML dir

In [2]:
import os
import re

# Configuration: set DRY_RUN = True to only print planned renames
DRY_RUN = False

data_dir = "../data/cleaned_html_2"
for fname in os.listdir(data_dir):
    old_path = os.path.join(data_dir, fname)
    if not os.path.isfile(old_path):
        continue

    # Attempt to extract a clean page-name from the filename
    # Common pattern in this dataset: <ip>_<port>_<page>.html
    m = re.search(r'^[^_]+_[^_]+_(.+)$', fname)
    if m:
        candidate = m.group(1)
    else:
        # Fallback: strip numeric prefix up to first underscore
        parts = fname.split('_', 1)
        candidate = parts[1] if len(parts) > 1 else fname

    # Ensure extension is .html (or preserve existing extension)
    base, ext = os.path.splitext(candidate)
    if ext.lower() not in ('.html', '.htm'):
        ext = os.path.splitext(fname)[1] or '.html'
    new_name = base + ext
    new_path = os.path.join(data_dir, new_name)

    # Avoid overwriting existing files: append a counter if needed
    if os.path.exists(new_path) and os.path.abspath(new_path) != os.path.abspath(old_path):
        i = 1
        while True:
            alt_name = f"{base}_{i}{ext}"
            alt_path = os.path.join(data_dir, alt_name)
            if not os.path.exists(alt_path):
                new_path = alt_path
                break
            i += 1

    print(f"Rename: {fname} -> {os.path.basename(new_path)}")
    if not DRY_RUN:
        try:
            os.rename(old_path, new_path)
        except Exception as e:
            print(f"Failed to rename {old_path}: {e}")


Rename: customer_attendance_day_off.html -> day_off.html
Rename: customer_attendance_employee_schedule.html -> employee_schedule.html
Rename: customer_attendance_leave.html -> leave.html
Rename: customer_attendance_leave_allowance.html -> leave_allowance.html
Rename: customer_attendance_overtime.html -> overtime.html
Rename: customer_attendance_schedule.html -> schedule.html
Rename: customer_attendance_status.html -> status.html
Rename: customer_cart.html -> cart.html
Rename: customer_changelog.html -> changelog.html
Rename: customer_dashboard.html -> dashboard.html
Rename: customer_device.html -> device.html
Rename: customer_device_api_sdk.html -> api_sdk.html
Rename: customer_device_push_server.html -> push_server.html
Rename: customer_employee.html -> employee.html
Rename: customer_employee_admin_multy_office.html -> admin_multy_office.html
Rename: customer_employee_app.html -> app.html
Rename: customer_employee_multy_office.html -> multy_office.html
Rename: customer_employee_no_cal

In [1]:
import os
import re

# Configuration: set DRY_RUN = True to only print planned renames
DRY_RUN = False

data_dir = "../data/json_html"
for fname in os.listdir(data_dir):
    old_path = os.path.join(data_dir, fname)
    if not os.path.isfile(old_path):
        continue

    # Attempt to extract a clean page-name from the filename
    # Common pattern in this dataset: <ip>_<port>_<page>.json
    m = re.search(r'^[^_]+_[^_]+_(.+)$', fname)
    if m:
        candidate = m.group(1)
    else:
        # Fallback: strip numeric prefix up to first underscore
        parts = fname.split('_', 1)
        candidate = parts[1] if len(parts) > 1 else fname

    # Ensure extension is .json (or preserve existing extension)
    base, ext = os.path.splitext(candidate)
    if ext.lower() not in ('.json',):
        ext = os.path.splitext(fname)[1] or '.json'
    new_name = base + ext
    new_path = os.path.join(data_dir, new_name)

    # Avoid overwriting existing files: append a counter if needed
    if os.path.exists(new_path) and os.path.abspath(new_path) != os.path.abspath(old_path):
        i = 1
        while True:
            alt_name = f"{base}_{i}{ext}"
            alt_path = os.path.join(data_dir, alt_name)
            if not os.path.exists(alt_path):
                new_path = alt_path
                break
            i += 1

    print(f"Rename: {fname} -> {os.path.basename(new_path)}")
    if not DRY_RUN:
        try:
            os.rename(old_path, new_path)
        except Exception as e:
            print(f"Failed to rename {old_path}: {e}")

Rename: 107.155.65.156_81_customer_attendance_day_off.json -> customer_attendance_day_off.json
Rename: 107.155.65.156_81_customer_attendance_employee_schedule.json -> customer_attendance_employee_schedule.json
Rename: 107.155.65.156_81_customer_attendance_leave.json -> customer_attendance_leave.json
Rename: 107.155.65.156_81_customer_attendance_leave_allowance.json -> customer_attendance_leave_allowance.json
Rename: 107.155.65.156_81_customer_attendance_overtime.json -> customer_attendance_overtime.json
Rename: 107.155.65.156_81_customer_attendance_schedule.json -> customer_attendance_schedule.json
Rename: 107.155.65.156_81_customer_attendance_status.json -> customer_attendance_status.json
Rename: 107.155.65.156_81_customer_cart.json -> customer_cart.json
Rename: 107.155.65.156_81_customer_changelog.json -> customer_changelog.json
Rename: 107.155.65.156_81_customer_dashboard.json -> customer_dashboard.json
Rename: 107.155.65.156_81_customer_device.json -> customer_device.json
Rename: 1

Rename JSON files - extract page name only

In [3]:
from bs4 import BeautifulSoup
from html.parser import HTMLParser
from pathlib import Path

# ── Lint helper: tracks warnings from Python's built-in HTML parser ──────────
class HTMLLinter(HTMLParser):
    def __init__(self):
        super().__init__()
        self.warnings = []

    def handle_error(self, message):          # only exists in strict parsers
        self.warnings.append(message)

    # Track unmatched tags via a simple open/close stack
    VOID_TAGS = {
        "area","base","br","col","embed","hr","img","input",
        "link","meta","param","source","track","wbr",
    }

    @classmethod
    def lint(cls, html_text):
        """Return a list of structural warnings for the given HTML string."""
        issues = []
        stack  = []
        parser = cls()

        original_start    = parser.handle_starttag
        original_end      = parser.handle_endtag
        original_startend = parser.handle_startendtag

        def _start(tag, attrs):
            if tag not in cls.VOID_TAGS:
                stack.append(tag)

        def _end(tag):
            if tag in cls.VOID_TAGS:
                return
            if stack and stack[-1] == tag:
                stack.pop()
            elif tag in stack:
                idx = len(stack) - 1 - stack[::-1].index(tag)
                skipped = stack[idx + 1:]
                issues.append(f"Unexpected </{tag}>; implicitly closed: {skipped}")
                del stack[idx:]
                stack.pop() if stack and stack[-1] == tag else None
            else:
                issues.append(f"Closing tag </{tag}> has no matching open tag")

        def _startend(tag, attrs):
            pass  # self-closing — fine

        parser.handle_starttag    = _start
        parser.handle_endtag      = _end
        parser.handle_startendtag = _startend
        parser.feed(html_text)

        for unclosed in stack:
            issues.append(f"Unclosed tag: <{unclosed}>")

        return issues

# ── Run lint + prettify on every scraped HTML ─────────────────────────────────
scraped_dir = Path("../data/cleaned_html_2")
html_files  = sorted(scraped_dir.glob("*.html"))

total_issues = 0

for f in html_files:
    raw = f.read_text(encoding="utf-8", errors="ignore")

    # 1. Lint
    issues = HTMLLinter.lint(raw)
    if issues:
        total_issues += len(issues)
        print(f"\n[LINT] {f.name}  ({len(issues)} issue(s))")
        for issue in issues[:10]:          # cap output per file
            print(f"       • {issue}")
        if len(issues) > 10:
            print(f"       … {len(issues) - 10} more")

    # 2. Format (prettify) and write back in-place
    pretty = BeautifulSoup(raw, "html.parser").prettify()
    f.write_text(pretty, encoding="utf-8")

print(f"\n{'─'*55}")
print(f"Linted & formatted {len(html_files)} files.")
print(f"Total lint issues found: {total_issues}")
if total_issues == 0:
    print("All files look clean!")



───────────────────────────────────────────────────────
Linted & formatted 47 files.
Total lint issues found: 0
All files look clean!
